In [6]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

C_2025, C_2075 = "#36B7B4", "#E6224B"

s2025 = [1.0,1.0,1.0,1.0,1.0,1.1,1.1,1.1,1.1,1.2, 1.1,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,
         1.2,1.3,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3, 1.3,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,
         1.3,1.3,1.3,1.3,1.3,1.3,1.2,1.2,1.1,1.2, 1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.3,
         1.3,1.3,1.3,1.2,1.2,1.1,1.1,1.1,1.0,1.0, 0.9,0.9,0.9,0.9,0.8,0.8,0.9,0.9,0.9,0.7,
         0.7,0.6,0.6,0.5,0.4,0.4,0.4,0.3,0.3,0.2, 0.2,0.2,0.1,0.1,0.1,0.1,0.0,0.0,0.0,0.0,
         0.0,0.0,0.0,0.0,0.0,0.0]
s2075 = [0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.8, 0.8,0.8,0.8,0.8,0.8,0.8,0.8,0.9,0.9,0.9,
         1.0,1.0,1.0,1.0,1.1,1.1,1.1,1.1,1.2,1.2, 1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,
         1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2,1.2, 1.2,1.2,1.2,1.3,1.3,1.3,1.3,1.3,1.3,1.4,
         1.3,1.3,1.4,1.4,1.4,1.3,1.3,1.3,1.3,1.2, 1.2,1.2,1.1,1.1,1.1,1.1,1.1,1.1,1.1,1.1,
         1.0,1.0,1.0,1.0,0.9,0.9,0.8,0.8,0.7,0.6, 0.6,0.5,0.5,0.4,0.3,0.3,0.2,0.2,0.1,0.1,
         0.1,0.0,0.0,0.0,0.0,0.0]

df = pd.DataFrame({"age": range(len(s2025)), "y2025": s2025, "y2075": s2075})
df["y2025"] = pd.Series(s2025).rolling(3, center=True, min_periods=1).mean()
df["y2075"] = pd.Series(s2075).rolling(3, center=True, min_periods=1).mean()
df["lo"] = np.minimum(df["y2025"], df["y2075"])
df["hi"] = np.maximum(df["y2025"], df["y2075"])
df["state"] = np.where(df["y2075"] >= df["y2025"], "More by 2075", "Fewer by 2075")

long = df.melt(id_vars="age", value_vars=["y2025", "y2075"],
               var_name="year", value_name="share")
long["year"] = long["year"].str.replace("y", "")

band = alt.Chart(df).mark_area(opacity=0.18).encode(
    x=alt.X("age:Q", scale=alt.Scale(domain=[0, 105], nice=False)),
    y=alt.Y("lo:Q", scale=alt.Scale(domain=[0, 1.6], nice=False)),
    y2="hi:Q",
    color=alt.Color("state:N",
        scale=alt.Scale(domain=["Fewer by 2075", "More by 2075"], range=[C_2025, C_2075]),
        legend=None))

lines = alt.Chart(long).mark_line(strokeWidth=2.6).encode(
    x=alt.X("age:Q", title="Age", scale=alt.Scale(domain=[0, 105], nice=False),
            axis=alt.Axis(values=list(range(0, 101, 10)), grid=False)),
    y=alt.Y("share:Q", title="Per cent",
            scale=alt.Scale(domain=[0, 1.6], nice=False),
            axis=alt.Axis(values=[x/10 for x in range(0, 17, 2)], format=".1f",
                          titleAngle=0, titleAlign="left", titleX=0, titleY=-8)),
    color=alt.Color("year:N",
        scale=alt.Scale(domain=["2025", "2075"], range=[C_2025, C_2075]),
        legend=alt.Legend(orient="none", legendX=75, legendY=15, direction="horizontal",
                          title=None, labelFontSize=12,
                          symbolType="stroke", symbolStrokeWidth=3)))

caption = alt.Title(
    text="Source: ONS, OBR",
    subtitle=[
        "Population share by single year of age, %. UK, 2025 vs 2075 projection.",
        "Teal = fewer people at that age by 2075 (younger ages); red = more (older ages).",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (band + lines)
    .resolve_scale(color="independent")
    .properties(width=700, height=260, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="2075_population", svg=True)
chart.save("2075_population.png", scale_factor=2.0)
chart


alt.LayerChart(...)